In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt

In [ ]:
adata_cancer = sc.read_h5ad('CC_GMM_groups_adata.h5ad')

In [ ]:
mito_genes = adata_cancer.var_names.str.startswith('mt-')
ribo_genes = adata_cancer.var_names.str.contains('^Rp[sl]')
adata_cancer = adata_cancer[:, ~(mito_genes | ribo_genes)].copy()

In [ ]:
sc.pp.normalize_total(adata_cancer, target_sum=1e4)
sc.pp.log1p(adata_cancer)
adata_cancer.layers['log1p'] = adata_cancer.X.copy()

In [ ]:
## DEG 
target_cells = ['ITC', 'PTC', 'MTC']
group_col = 'Kinetics_State' 


pval_cutoff = 0.05
logfc_cutoff = 0.58

# sub_type
for cctype in target_cells:

    
    adata_sub = adata_cancer[adata_cancer.obs['CCtype'] == cctype].copy()
    adata_sub = adata_sub[adata_sub.obs[group_col].isin(['High', 'Low'])]
    
    if len(adata_sub.obs[group_col].unique()) < 2:
        print(f"PASS")
        continue

# Wilcoxon
    sc.tl.rank_genes_groups(
        adata_sub, 
        groupby=group_col, 
        groups=['High'],     
        reference='Low',     
        method='t-test_overestim_var', 
        use_raw=False,      
        key_added='deg_High_vs_Low'
    )
    
    result_df = sc.get.rank_genes_groups_df(adata_sub, group='High', key='deg_High_vs_Low')
    high_degs = result_df[(result_df['pvals_adj'] < pval_cutoff) & (result_df['logfoldchanges'] > logfc_cutoff)]
    low_degs = result_df[(result_df['pvals_adj'] < pval_cutoff) & (result_df['logfoldchanges'] < -logfc_cutoff)]
    

    # save
    high_degs['names'].to_csv(f"{cctype}_High_DEGs_List.csv", index=False, header=False)
    low_degs['names'].to_csv(f"{cctype}_Low_DEGs_List.csv", index=False, header=False)
    high_degs.to_csv(f"{cctype}_High_DEGs_Full.csv", index=False)
    low_degs.to_csv(f"{cctype}_Low_DEGs_Full.csv", index=False)
    
    print(f"{cctype}: Find: High:{len(high_degs)}, Low:{len(low_degs)}")

print("\nFinish!")

In [ ]:
def run_go_enrichment(deg_csv_path, output_csv_path):
    
    df_genes = pd.read_csv(deg_csv_path, header=None)
    gene_list = df_genes[0].tolist()
    
    # 2.GO
    enr = gp.enrichr(gene_list=gene_list,
                     gene_sets='GO_Biological_Process_2021', 
                     organism='mouse', 
                     outdir=None)   
    results = enr.results  
    #  P < 0.05
    sig_results = results[results['Adjusted P-value'] < 0.05].copy() 
    # save csv
    sig_results.to_csv(output_csv_path, index=False)


In [ ]:

run_go_enrichment("ITC_Low_DEGs_List.csv", "ITC_Low_GO.csv")
run_go_enrichment("PTC_Low_DEGs_List.csv", "PTC_Low_GO.csv")
run_go_enrichment("MTC_Low_DEGs_List.csv", "MTC_Low_GO.csv")

In [ ]:
run_go_enrichment("PTC_High_DEGs_List.csv", "PTC_High_GO.csv")
run_go_enrichment("ITC_High_DEGs_List.csv", "ITC_High_GO.csv")
run_go_enrichment("MTC_High_DEGs_List.csv", "MTC_High_GO.csv")

In [ ]:
def plot_bidirectional_go(sub_cell,high_csv, low_csv, output_pdf="Fig_PTC_Bidirectional_GO.pdf", top_n=5):

    df_high = pd.read_csv(high_csv)
    df_low = pd.read_csv(low_csv)

    df_high = df_high.sort_values('Adjusted P-value').head(top_n).copy()
    df_low = df_low.sort_values('Adjusted P-value').head(top_n).copy()
    
    df_high['Plot_Value'] = -np.log10(df_high['Adjusted P-value'])
    df_high['Color'] = '#e74c3c' 
    df_high['Group'] = 'High'

    df_low['Plot_Value'] = -np.log10(df_low['Adjusted P-value']) * -1
    df_low['Color'] = '#3498db'  
    df_low['Group'] = 'Low'
    
    for df in [df_high, df_low]:
        df['Term'] = df['Term'].apply(lambda x: x.split('(GO:')[0].strip())
        
    df_plot = pd.concat([df_low.sort_values('Plot_Value', ascending=True), 
                         df_high.sort_values('Plot_Value', ascending=True)], axis=0).reset_index(drop=True)
    
    fig, ax = plt.subplots(figsize=(6, 0.4 * len(df_plot) + 1.5))

    bars = ax.barh(
        y=df_plot['Term'], 
        width=df_plot['Plot_Value'], 
        color=df_plot['Color'],
        edgecolor='white',   
        linewidth=1,
        height=0.85        
    )
    
    ax.grid(False)
    ax.axvline(0, color='black', linewidth=1)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)   
    ax.spines['bottom'].set_linewidth(1.2)

    ax.tick_params(axis='x', labelsize=11, width=1.2)
    ax.tick_params(axis='y', labelsize=11, length=0)
    
    ax.set_xlabel("Log.q.value", fontsize=13, fontweight='bold', labelpad=8)
    ax.set_title(sub_cell, fontsize=14, fontweight='bold', pad=15, loc='center') 
    
    plt.tight_layout()
    plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)

plot_bidirectional_go('ITC',"ITC_High_GO.csv", "ITC_Low_GO.csv","Fig_ITC_Bidirectional_GO.pdf", top_n=5)